# Extension 05 Differentiable optimisation

In [ ]:
%pip install -q numpy session-py==0.310.0 session-jupyter==0.2.0

# Question

**How do we differentiate through the linear solve to tune $q$ for target heights?**

```
n42 ─ n43 ─ n44 ─ n45 ─ n46 ─ n47 ─ n48
 │     │     │     │     │     │     │
n35 ─ n36 ─ n37 ─ n38 ─ n39 ─ n40 ─ n41
 │     │     │     │     │     │     │
n28 ─ n29 ─ n30 ─ n31 ─ n32 ─ n33 ─ n34
 │     │     │     │     │     │     │
n21 ─ n22 ─ n23 ─ n24 ─ n25 ─ n26 ─ n27
 │     │     │     │     │     │     │
n14 ─ n15 ─ n16 ─ n17 ─ n18 ─ n19 ─ n20
 │     │     │     │     │     │     │
n7  ─ n8  ─ n9  ─ n10 ─ n11 ─ n12 ─ n13
 │     │     │     │     │     │     │
n0  ─ n1  ─ n2  ─ n3  ─ n4  ─ n5  ─ n6
```

**Given:** the $6 \times 6$ cable net pinned at its four corners at $z = 0\text{ m}$ under downward load ${\color{#2e9e4a} p_z = -0.3\text{ kN}}$, and desired target heights $z^*_k$ at selected free nodes.

**Unknown:** force densities $q > 0$ that steer the net onto the target heights.

**Sensitivity:** differentiating $\left(C_N^{T}\,Q\,C_N\right) z_N = p_z - C_N^{T}\,Q\,C_F\,z_F$ with respect to $q$ gives the exact Jacobian in one linear solve with the same matrix $A = C_N^{T}\,Q\,C_N$:

$$ A\,\frac{\partial z_N}{\partial q} = -C_N^{T}\,\operatorname{diag}(C\,z) $$

<div align="center">
  <img src="https://raw.githubusercontent.com/BRG-teaching/Teaching/main/CAS/logos/fd_coordinate_difference.svg"
       style="width:100%; height:auto;"
       alt="Coordinate differences along bars coupling force density changes to node displacements" />
  <p style="text-align:center;">
    <em>Differentiating the equilibrium equation gives exact gradients with respect to every bar force density.</em>
  </p>
</div>

**References:**

- **AsapOptim**, https://github.com/keithjlee/AsapOptim.jl
- **JAX FDM**, https://github.com/arpastrana/jax_fdm
- **Introduction to Differentiable Form Finding with JAX FDM**, https://github.com/arpastrana/jaxfdm-workshop
- **Neural FDM**, https://github.com/PrincetonLIPS/neural_fdm
- **VAE FDM**, https://github.com/efradeca/vae-fdm

<hr style="border: none; border-top: 2px solid #555; margin: 2em 0;">

# Network and Exact Sensitivity

In [1]:
import numpy as np

# nodes and bars of the 6 x 6 grid
n = 7
x, y = np.meshgrid(range(n), range(n))
x = x.flatten().astype(float)
y = y.flatten().astype(float)
z = np.zeros(n * n)

edges = [(k, k+1) for k in range(n*n) if x[k] < n-1] + \
        [(k, k+n) for k in range(n*n) if y[k] < n-1]
C = np.zeros((len(edges), n * n))
for e, (a, b) in enumerate(edges):
    C[e, a], C[e, b] = 1, -1

fixed = [0, n-1, n*(n-1), n*n - 1]
free  = [k for k in range(n*n) if k not in fixed]
C_N, C_F = C[:, free], C[:, fixed]
p_z = -0.3 * np.ones(len(free))


def solve_and_grad(q):
    """Forward solve (# equation 6.32) and exact sensitivity dz_N / dq."""
    Q = np.diag(q)
    A = C_N.T @ Q @ C_N
    z_cur = np.zeros(n * n)
    z_cur[free] = np.linalg.solve(A, p_z - C_N.T @ Q @ C_F @ z_cur[fixed])
    dz_dq = np.linalg.solve(A, -C_N.T @ np.diag(C @ z_cur))
    return z_cur, dz_dq


# check analytical sensitivity against central finite differences on bar e0
q0 = 1.5 * np.ones(len(edges))
z_start, dz_dq = solve_and_grad(q0)
dq = np.zeros(len(edges))
dq[0] = 1e-6
fd_e0 = (solve_and_grad(q0 + dq)[0][free] - solve_and_grad(q0 - dq)[0][free]) / (2e-6)
print("max error vs finite differences =", f"{np.abs(dz_dq[:, 0] - fd_e0).max():.2e}")

max error vs finite differences = 1.40e-09


# Gradient Descent on Force Densities

Lift the middle node ${\color{#d62828} n_{24}}$ from $-2.10\text{ m}$ to $-1.00\text{ m}$ and the four mid edge nodes ${\color{#d62828} n_3, n_{21}, n_{27}, n_{45}}$ to $-0.45\text{ m}$ by stepping $q$ down the exact gradient of $L(q) = \frac{1}{2}\|z_{\text{targets}} - z^*\|^2$.

In [2]:
targets = [24, 3, 21, 27, 45]
target_rows = [free.index(k) for k in targets]
z_goal = np.array([-1.00, -0.45, -0.45, -0.45, -0.45])

q = q0.copy()
for step in range(60):
    z_cur, dz_dq = solve_and_grad(q)
    err = z_cur[targets] - z_goal
    grad = dz_dq[target_rows].T @ err
    q = np.clip(q - 3.0 * grad, 0.2, 20.0)

z_opt, _ = solve_and_grad(q)

# solve x and y for the optimised q
Q = np.diag(q)
A = C_N.T @ Q @ C_N
x_opt, y_opt = x.copy(), y.copy()
x_opt[free] = np.linalg.solve(A, -C_N.T @ Q @ C_F @ x_opt[fixed])
y_opt[free] = np.linalg.solve(A, -C_N.T @ Q @ C_F @ y_opt[fixed])

print("initial target heights   =", np.round(z_start[targets], 3), "m")
print("optimised target heights =", np.round(z_opt[targets], 3), "m")
print("goal heights             =", z_goal, "m")

initial target heights   = [-2.098 -1.81  -1.81  -1.81  -1.81 ] m
optimised target heights = [-1.013 -0.586 -0.586 -0.586 -0.586] m
goal heights             = [-1.   -0.45 -0.45 -0.45 -0.45] m


# Viewer

Gray: uniform $q = 1.5\text{ kN/m}$ net. Black: optimised net hitting all five target heights, with darker bars showing where the gradient increased $q$.

In [3]:
from session_py import Color, Matrix, Session
from session_jupyter import show

# the colours of the figures: free nodes red, fixed nodes blue, reference grey
RED  = Color(0.84, 0.16, 0.16, 1.0)
BLUE = Color(0.16, 0.44, 0.86, 1.0)
GRAY = Color(0.70, 0.70, 0.70, 1.0)

# initial x and y for uniform q0
Q0 = np.diag(q0)
A0 = C_N.T @ Q0 @ C_N
x0, y0 = x.copy(), y.copy()
x0[free] = np.linalg.solve(A0, -C_N.T @ Q0 @ C_F @ x0[fixed])
y0[free] = np.linalg.solve(A0, -C_N.T @ Q0 @ C_F @ y0[fixed])

f = q * np.sqrt((C @ x_opt)**2 + (C @ y_opt)**2 + (C @ z_opt)**2)
colors = [Color(0.65 - 0.65 * fv / f.max(), 0.65 - 0.65 * fv / f.max(), 0.65 - 0.65 * fv / f.max(), 1.0) for fv in f]

session = Session("fdm_ext05_optimised_net")
session.add_network(Matrix.from_rows(C.tolist()), Matrix.from_rows(np.column_stack([x0, y0, z_start]).tolist()), [GRAY] * len(edges), [1.5] * len(edges))
session.add_network(Matrix.from_rows(C.tolist()), Matrix.from_rows(np.column_stack([x_opt, y_opt, z_opt]).tolist()), colors, [3.0] * len(edges))

for net in range(2):
    for k in range(n * n):
        pt = session.objects.points[net * n * n + k]
        pt.pointcolor = BLUE if k in fixed else RED
        pt.width = 13.0 if k in fixed else 6.5

show(session, "fdm_ext05_optimised_net")

# Observation

Because `# equation 6.32` is linear, differentiating through equilibrium needs only one solve with the same matrix $A$. Automatic differentiation libraries (`jax_fdm`, `AsapOptim.jl`) apply this exact rule in reverse mode to optimise thousands of bars or train neural form finders.